# Rebuild `uni.mglyph`

Generates `data/uni.mglyph`: gray triangles, splits `"0"` (train) and `"1"` (test), 5000 samples each.


In [ ]:
import random
import mglyph as mg
from pathlib import Path
from zipfile import ZipFile

import mglyph as mg
import numpy as np

from mglyph_ml.dataset.export import create_dataset
from mglyph_ml.experiment.e1.util import ManifestSampleShape

In [ ]:
from mglyph_ml.dataset.export import Drawer


def generate_dataset(
    name: str, path: Path, drawer: Drawer, seed: int | None = None, samples_per_x: int = 50
) -> None:
    # Ensure the output directory exists
    path.parent.mkdir(parents=True, exist_ok=True)

    # Remove existing file so generation is always fresh and deterministic.
    if path.exists():
        path.unlink()
        print(f"Removed existing {path}")

    np_gen = np.random.default_rng(seed)
    random.seed(seed)

    dataset = create_dataset(name=name)
    total_samples_shape = samples_per_x * 100

    xvalues_train = np_gen.uniform(0.0, 100.0, total_samples_shape)
    xvalues_train = np.append(xvalues_train, [0.0, 100.0])
    xvalues_train.sort()

    xvalues_test = np_gen.uniform(0.0, 100.0, 2000)
    xvalues_test = np.append(xvalues_test, [0.0, 100.0])
    xvalues_test.sort()

    for x in xvalues_train:
        dataset.add_sample(drawer, x, split="0")

    for x in xvalues_test:
        dataset.add_sample(drawer, x, split="1")

    dataset.export(path)
    print(f"Generated: {path}")

In [ ]:
def triangle(x: float, canvas: mg.Canvas):
    canvas.tr.scale(mg.lerp(x, 0.05, 0.95))
    canvas.polygon([canvas.bottom_left, canvas.bottom_right, canvas.top_center], color="gray")


def star(x: float, canvas: mg.Canvas, bordercolor, fillcolor, linewidth) -> None:
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * np.pi / 5, radius))
        vertices.append(
            mg.orbit(
                canvas.center,
                (segment + 0.5) * 2 * np.pi / 5,
                np.cos(2 * np.pi / 5) / np.cos(np.pi / 5) * radius,
            )
        )

    canvas.polygon(vertices, linecap="round", style="fill", color=fillcolor)
    canvas.polygon(vertices, width=linewidth, linecap="round", style="stroke", color=bordercolor)


def simple_star() -> Drawer:
    bordercolor = "navy"
    fillcolor = "white"
    linewidth = "70p"
    return lambda x, canvas: star(x, canvas, bordercolor, fillcolor, linewidth)

mg.show(simple_star()) # type: ignore

generate_dataset("Simple Star", Path("data/simple-star.mglyph"), simple_star(), samples_per_x=100)